# NFL Game Performance Analysis - Final Project

## Project Overview

We are writing to propose our final project topic. We plan to analyze NFL game performance data using publicly available datasets from Pro Football Reference and NFLverse. 

The datasets include game statistics, weather conditions, venue information, and scheduling details across multiple NFL seasons, providing rich opportunities for visualization.

## Why NFL Data Insights Are Important

NFL data insights are crucial for several reasons:

1. **Strategic Decision Making**: Understanding how external factors like weather and venue characteristics impact game outcomes helps teams, coaches, and analysts make more informed strategic decisions.

2. **Fan Engagement**: Data-driven visualizations help fans better understand the game, appreciate the complexity of factors affecting performance, and engage more deeply with the sport.

3. **Performance Optimization**: By identifying patterns in how scheduling, weather, and venue conditions affect performance, teams can optimize their preparation and game plans.

4. **Predictive Analytics**: These insights contribute to building more accurate predictive models for game outcomes, which is valuable for fantasy football, betting analysis, and team management.

5. **Scientific Understanding**: Analyzing large-scale sports data provides insights into human performance under varying conditions, contributing to broader understanding of athletic performance and decision-making.

## Proposed Research Questions

### 1. How do weather conditions (temperature, precipitation, wind) impact game scoring and performance?


## Data Collection

We'll use two data collection functions to gather NFL data:
1. `download_nfl_data()` - Downloads schedules, team information, and creates an analysis-ready merged dataset
2. `collect_weather_data()` - Collects weather-specific data for all games and outdoor-only games


In [1]:
import sys
import os
sys.path.append('scripts')

from download_nfl_data import download_nfl_data
from weather_data import collect_weather_data, enrich_games_with_meteostat, add_gametime_from_nfl_data
import pandas as pd
import numpy as np


### Download NFL Schedules and Team Data

This function downloads game schedules, team information, and creates a merged dataset with calculated metrics like point differential and home win indicators.


In [2]:
# Download NFL data for seasons 2020-2024
seasons = [2020, 2021, 2022, 2023, 2024]
nfl_data = download_nfl_data(seasons)

# Verify data was received
print(f"Data shape: {nfl_data.shape}")
print(f"\nColumns: {list(nfl_data.columns)}")
print(f"\nFirst few rows:")
nfl_data.head()


Data shape: (1408, 50)

Columns: ['game_id', 'season', 'game_type', 'week', 'gameday', 'weekday', 'gametime', 'away_team', 'away_score', 'home_team', 'home_score', 'location', 'result', 'total', 'overtime', 'old_game_id', 'gsis', 'nfl_detail_id', 'pfr', 'pff', 'espn', 'ftn', 'away_rest', 'home_rest', 'away_moneyline', 'home_moneyline', 'spread_line', 'away_spread_odds', 'home_spread_odds', 'total_line', 'under_odds', 'over_odds', 'div_game', 'roof', 'surface', 'temp', 'wind', 'away_qb_id', 'home_qb_id', 'away_qb_name', 'home_qb_name', 'away_coach', 'home_coach', 'referee', 'stadium_id', 'stadium', 'team_abbr', 'team_name', 'point_differential', 'home_win']

First few rows:


,game_id,season,game_type,week,gameday,weekday,gametime,away_team,away_score,home_team,...,home_qb_name,away_coach,home_coach,referee,stadium_id,stadium,team_abbr,team_name,point_differential,home_win
0,2020_01_HOU_KC,2020,REG,1,2020-09-10,Thursday,20:20,HOU,20,KC,...,Patrick Mahomes,Bill O'Brien,Andy Reid,Clete Blakeman,KAN00,Arrowhead Stadium,KC,Kansas City Chiefs,14,1
1,2020_01_SEA_ATL,2020,REG,1,2020-09-13,Sunday,13:00,SEA,38,ATL,...,Matt Ryan,Pete Carroll,Dan Quinn,Shawn Hochuli,ATL97,Mercedes-Benz Stadium,ATL,Atlanta Falcons,-13,0
2,2020_01_CLE_BAL,2020,REG,1,2020-09-13,Sunday,13:00,CLE,6,BAL,...,Lamar Jackson,Kevin Stefanski,John Harbaugh,Ronald Torbert,BAL00,M&T Bank Stadium,BAL,Baltimore Ravens,32,1
3,2020_01_NYJ_BUF,2020,REG,1,2020-09-13,Sunday,13:00,NYJ,17,BUF,...,Josh Allen,Adam Gase,Sean McDermott,Shawn Smith,BUF00,New Era Field,BUF,Buffalo Bills,10,1
4,2020_01_LV_CAR,2020,REG,1,2020-09-13,Sunday,13:00,LV,34,CAR,...,Teddy Bridgewater,Jon Gruden,Matt Rhule,Brad Allen,CAR00,Bank of America Stadium,CAR,Carolina Panthers,-4,0


### Collect Weather Data

This function collects weather-specific data, filtering for outdoor games that have actual weather conditions (temperature, wind, etc.).


In [3]:
# Collect weather data
all_games, outdoor_games = collect_weather_data(seasons)

# Verify data was received
print(f"All games shape: {all_games.shape}")
print(f"Outdoor games shape: {outdoor_games.shape}")
print(f"\nOutdoor games represent {len(outdoor_games)/len(all_games)*100:.1f}% of all games")
print(f"\nAll games columns: {list(all_games.columns)}")


All games shape: (1408, 18)
Outdoor games shape: (814, 18)

Outdoor games represent 57.8% of all games

All games columns: ['game_id', 'season', 'week', 'gameday', 'weekday', 'home_team', 'away_team', 'home_score', 'away_score', 'temp', 'wind', 'roof', 'surface', 'stadium', 'gametime', 'total_points', 'point_differential', 'home_win']


### Enrich Weather Data with Meteostat API
To get accurate weather dependency on the game, just temperature and wind is not enough. So we will be using Meteostat API to get additional weather data based on the date, time and location of each football game. 

We'll enrich the weather data using the Meteostat API to get additional weather metrics:
- Temperature (Celsius)
- Dewpoint (Celsius)
- Humidity (percentage)
- Precipitation (mm)
- Wind speed (m/s)

In [4]:
def create_weather_data(outdoor_games, nfl_data):
    # Merge gametime from nfl_data into outdoor_games
    outdoor_games = add_gametime_from_nfl_data(outdoor_games, nfl_data)

    # Ensure gameday is datetime for outdoor games
    outdoor_games['gameday'] = pd.to_datetime(outdoor_games['gameday'])

    # Enrich outdoor games with Meteostat data
    print(f"Enriching {len(outdoor_games)} outdoor games with Meteostat API...")
    print("This may take several minutes depending on the number of games...")
    enriched_outdoor_games = enrich_games_with_meteostat(
        outdoor_games, 
        stadium_col='stadium', 
        gameday_col='gameday',
        gametime_col='gametime'
    )
    # Drop rows where key Meteostat weather columns are missing
    # This keeps rows even if some non-critical columns have nulls
    met_weather_cols = ['met_temp_C', 'met_humidity_pct', 'met_windspeed_mps']
    enriched_outdoor_games = enriched_outdoor_games.dropna(subset=met_weather_cols)

    print(f"\nEnriched data shape: {enriched_outdoor_games.shape}")
    print(f"New columns added: {[col for col in enriched_outdoor_games.columns if col.startswith('met_')]}")

    # save the enriched data
    enriched_outdoor_games.to_csv('data/enriched_outdoor_games.csv', index=False)
    return enriched_outdoor_games


# if enriched_outdoor_games.csv exists, skip the enrichment
if os.path.exists('data/enriched_outdoor_games.csv'):
    enriched_outdoor_games = pd.read_csv('data/enriched_outdoor_games.csv')
else:
    enriched_outdoor_games = create_weather_data(outdoor_games, nfl_data)


In [5]:
# Check the enriched data
print("Sample of enriched outdoor games weather data:")
enriched_outdoor_games[['game_id', 'stadium', 'gameday', 'gametime', 'met_temp_C', 'met_dewpoint_C', 
                'met_humidity_pct', 'met_prcp_mm', 'met_windspeed_mps', 'met_lat', 'met_lon']].head(10)

Sample of enriched outdoor games weather data:


,game_id,stadium,gameday,gametime,met_temp_C,met_dewpoint_C,met_humidity_pct,met_prcp_mm,met_windspeed_mps,met_lat,met_lon
0,2020_01_HOU_KC,Arrowhead Stadium,2020-09-10,20:20,13.300000,12.20,93.000000,0.000000,11.200000,39.048936,-94.483999
1,2020_01_CLE_BAL,M&T Bank Stadium,2020-09-13,13:00,25.933333,17.70,60.666667,0.000000,9.400000,39.277973,-76.622665
2,2020_01_NYJ_BUF,New Era Field,2020-09-13,13:00,20.766667,17.20,80.333333,0.433333,28.433333,42.773733,-78.786880
3,2020_01_LV_CAR,Bank of America Stadium,2020-09-13,13:00,27.200000,18.90,60.666667,0.000000,8.066667,35.225774,-80.852828
4,2020_01_MIA_NE,Gillette Stadium,2020-09-13,13:00,22.966667,13.10,54.000000,0.000000,14.900000,42.090888,-71.264173
5,2020_01_LAC_CIN,Paul Brown Stadium,2020-09-13,16:05,27.200000,17.35,55.000000,0.000000,7.400000,39.095464,-84.516049
6,2020_01_ARI_SF,Levi's Stadium,2020-09-13,16:25,21.400000,13.30,60.000000,0.000000,9.400000,37.402987,-121.969649
7,2020_01_PIT_NYG,MetLife Stadium,2020-09-14,19:15,20.000000,7.50,44.500000,0.000000,18.400000,40.813507,-74.074344
8,2020_01_TEN_DEN,Empower Field at Mile High,2020-09-14,22:10,20.300000,2.35,30.500000,0.000000,14.850000,39.743952,-105.019223
9,2020_02_CIN_CLE,FirstEnergy Stadium,2020-09-17,20:20,18.300000,8.00,51.000000,0.000000,16.600000,41.506056,-81.699712


# Data Summary
The datasets have been processed and enriched as follows:
- Full NFL schedules, scores, team info, and additional calculated metrics (point differential, total points, home win indicator) are loaded.
- Weather columns from the NFL's own data (temperature, wind, etc.) are included for all games.
- All outdoor games are further enriched with detailed historical weather data from the Meteostat API:
    - Meteostat columns: temperature (met_temp_C), dew point (met_dewpoint_C), humidity (met_humidity_pct), precipitation (met_prcp_mm), wind speed (met_windspeed_mps), latitude & longitude of stadium, source time.
- Stadium details (roof type, surface, and name) are merged.
- Games where Meteostat weather data could not be obtained are filtered from the enriched outdoor dataset.
- Enriched datasets are saved as CSV and reloaded as needed to avoid redundant API calls.
